# Training del classificatore — Marine AI

Fine-tuning di un backbone pre-addestrato sulle 196 classi finali (dopo fusione duplicati) dello split `intra_class` di ORCA.

Riusa `species_environmental_coverage.csv` (generato dal notebook 02) come fonte della mappa `common_name -> canonical_label`, cosi' non ricalcoliamo la logica di fusione da zero.

> Nota sull'hardware: il codice usa `torch.device("cuda")`, l'alias standard di PyTorch per la GPU disponibile — con una build compilata per ROCm, questo alias punta correttamente anche a GPU AMD, non solo NVIDIA. Se `torch.cuda.is_available()` restituisce `False`, verificare di avere installata la build di PyTorch corretta per il proprio hardware (CUDA per NVIDIA, ROCm per AMD) da pytorch.org/get-started/locally.

## 0. Setup

In [1]:
# pip install torch torchvision pandas pillow tqdm scikit-learn --quiet
# per ROCm: segui https://pytorch.org/get-started/locally/ (seleziona ROCm, non CUDA)

import json
import time
from pathlib import Path
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report

def find_project_root(marker=".gitignore"):
    path = Path.cwd()
    while path != path.parent:
        if (path / marker).exists():
            return path
        path = path.parent
    raise FileNotFoundError(f"Marker '{marker}' non trovato risalendo da {Path.cwd()}")

PROJECT_ROOT = find_project_root()
ORCA_DIR = PROJECT_ROOT / "data" / "orca"
SPLIT_DIR = ORCA_DIR / "split_annotations"
IMAGES_DIR = ORCA_DIR / "images"
COVERAGE_CSV = PROJECT_ROOT / "docs" / "species_environmental_coverage.csv"  # generato dal notebook 02
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("ATTENZIONE: nessuna GPU rilevata, il training su CPU sara' molto piu' lento.")

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


Device: cuda


## 1. Mappa delle classi finali (196, dopo fusione duplicati)

In [2]:
coverage_df = pd.read_csv(COVERAGE_CSV)

# common_name -> canonical_label (identita' per le classi non coinvolte in fusioni)
LABEL_MAP = dict(zip(coverage_df["common_name"], coverage_df["canonical_label"]))

canonical_labels = sorted(set(LABEL_MAP.values()))
label_to_idx = {label: i for i, label in enumerate(canonical_labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

NUM_CLASSES = len(canonical_labels)
print(f"Classi finali: {NUM_CLASSES}")
assert NUM_CLASSES == 196, f"Attesi 196, trovati {NUM_CLASSES} — verifica coverage_df prima di continuare"

Classi finali: 196


## 2. Dataset PyTorch

Carica gli split ufficiali `intra_class_train`/`intra_class_val_seen`, filtra sulle classi selezionate, applica la mappa canonica.

In [3]:
def load_split_records(path, label_map, images_dir):
    with open(path) as f:
        coco = json.load(f)

    images_by_id = {img["id"]: img["file_name"] for img in coco["images"]}
    cat_by_id = {c["id"]: c["name"] for c in coco["categories"]}

    records = []
    seen_images = set()
    for ann in coco["annotations"]:
        if ann["label"] not in (0, 2):
            continue
        common_name = cat_by_id.get(ann["category_id"])
        if common_name not in label_map:
            continue  # fuori dalle 200 specie selezionate a monte (soglia 20 img/specie)
        img_id = ann["image_id"]
        if img_id in seen_images:
            continue  # una sola annotazione per immagine, evitiamo duplicati nel dataset di training
        seen_images.add(img_id)
        file_name = images_by_id[img_id]
        records.append({
            "image_path": str(images_dir / Path(file_name).name),
            "canonical_label": label_map[common_name],
        })
    return pd.DataFrame(records)

train_records = load_split_records(SPLIT_DIR / "intra_class_train.json", LABEL_MAP, IMAGES_DIR)
val_records = load_split_records(SPLIT_DIR / "intra_class_val_seen.json", LABEL_MAP, IMAGES_DIR)

train_records["label_idx"] = train_records["canonical_label"].map(label_to_idx)
val_records["label_idx"] = val_records["canonical_label"].map(label_to_idx)

print(f"Immagini di training: {len(train_records)}")
print(f"Immagini di validazione: {len(val_records)}")
print(f"Classi presenti nel train: {train_records['canonical_label'].nunique()} / {NUM_CLASSES}")

Immagini di training: 5863
Immagini di validazione: 1650
Classi presenti nel train: 196 / 196


In [4]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class MarineSpeciesDataset(Dataset):
    def __init__(self, records_df, transform):
        self.records = records_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        img = self.transform(img)
        return img, row["label_idx"]

train_dataset = MarineSpeciesDataset(train_records, train_transform)
val_dataset = MarineSpeciesDataset(val_records, val_transform)

# WeightedRandomSampler: compensa lo sbilanciamento pesando di piu' le classi rare durante il training
class_counts = Counter(train_records["label_idx"])
sample_weights = train_records["label_idx"].map(lambda i: 1.0 / class_counts[i]).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

/home/ubuntu/marine_ai/.venv_rocm/lib/python3.12/site-packages/torch/utils/data/sampler.py:264: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  weights_tensor = torch.as_tensor(weights, dtype=torch.double)


## 3. Modello — fine-tuning di EfficientNet-B0

Backbone pre-addestrato su ImageNet, testa di classificazione sostituita per le 196 classi. Con ~30-45 immagini/specie in media, il training da zero non e' un'opzione: il transfer learning e' necessario, non solo preferibile.

In [5]:
model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
EPOCHS = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 4. Training loop

In [6]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

history = []
best_val_acc = 0.0
BEST_MODEL_PATH = MODELS_DIR / "marine_classifier_best.pt"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_preds, val_labels = run_epoch(model, val_loader, criterion)
    scheduler.step()

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc})
    print(f"Epoch {epoch}/{EPOCHS} — train_loss {train_loss:.3f} acc {train_acc:.3f} | val_loss {val_loss:.3f} acc {val_acc:.3f} ({time.time()-t0:.0f}s)")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "idx_to_label": idx_to_label,
            "val_acc": val_acc,
            "epoch": epoch,
        }, BEST_MODEL_PATH)
        print(f"  -> nuovo best model salvato (val_acc={val_acc:.3f})")

with open(MODELS_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

  0%|          | 0/184 [00:00<?, ?it/s]

/home/ubuntu/marine_ai/.venv_rocm/lib/python3.12/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  0%|          | 0/52 [00:00<?, ?it/s]

Epoch 1/15 — train_loss 3.985 acc 0.287 | val_loss 3.328 acc 0.350 (621s)
  -> nuovo best model salvato (val_acc=0.350)


  0%|          | 0/184 [00:00<?, ?it/s]

...

  0%|          | 0/52 [00:00<?, ?it/s]

Epoch 14/15 — train_loss 1.045 acc 0.980 | val_loss 2.460 acc 0.545 (516s)


  0%|          | 0/184 [00:00<?, ?it/s]

  0%|          | 0/52 [00:00<?, ?it/s]

Epoch 15/15 — train_loss 1.039 acc 0.980 | val_loss 2.459 acc 0.546 (515s)


## 5. Valutazione stratificata per fascia di supporto

Non solo l'accuratezza media: qui verifichiamo se le classi con poche immagini di training se la cavano peggio di quelle piu' numerose.

In [7]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

_, _, val_preds, val_labels = run_epoch(model, val_loader, criterion)

train_support = train_records["label_idx"].value_counts().to_dict()

report = classification_report(val_labels, val_preds, output_dict=True, zero_division=0)

rows = []
for idx_str, metrics in report.items():
    if not idx_str.isdigit():
        continue
    idx = int(idx_str)
    rows.append({
        "canonical_label": idx_to_label[idx],
        "train_support": train_support.get(idx, 0),
        "val_precision": metrics["precision"],
        "val_recall": metrics["recall"],
        "val_f1": metrics["f1-score"],
    })
per_class_df = pd.DataFrame(rows)

bins = [0, 25, 40, 60, float("inf")]
bin_labels = ["20-25 img", "26-40 img", "41-60 img", "60+ img"]
per_class_df["support_bucket"] = pd.cut(per_class_df["train_support"], bins=bins, labels=bin_labels)

bucket_summary = per_class_df.groupby("support_bucket", observed=True).agg(
    n_classi=("canonical_label", "count"),
    f1_medio=("val_f1", "mean"),
    recall_medio=("val_recall", "mean"),
).reset_index()

print(f"Accuratezza complessiva sul val set: {best_val_acc:.3f}\n")
print("Metriche per fascia di supporto (immagini di training per classe):")
bucket_summary

  0%|          | 0/52 [00:00<?, ?it/s]

Accuratezza complessiva sul val set: 0.549

Metriche per fascia di supporto (immagini di training per classe):


,support_bucket,n_classi,f1_medio,recall_medio
0,20-25 img,91,0.566480,0.639887
1,26-40 img,91,0.568632,0.638780
2,41-60 img,10,0.730454,0.819496
3,60+ img,4,0.370984,0.333517


In [8]:
per_class_df.to_csv(MODELS_DIR / "per_class_metrics.csv", index=False)
bucket_summary.to_csv(MODELS_DIR / "support_bucket_summary.csv", index=False)
print("Salvati per_class_metrics.csv e support_bucket_summary.csv in models/ — utili per il grafico da mettere nel report finale")

Salvati per_class_metrics.csv e support_bucket_summary.csv in models/ — utili per il grafico da mettere nel report finale
